# 02 Chebyshev 与 Legendre 级数逼近

本节讨论正交多项式逼近。第二章已经使用过 Chebyshev 节点来缓解 Runge 现象；在第三章中，Chebyshev 多项式被看作函数逼近的基函数。

两条主线分别是：

* Chebyshev 多项式适合区间逼近、端点控制和后续谱方法；
* Legendre 多项式自然来自普通平方范数下的正交投影。

二者都定义在标准区间 $[-1,1]$ 上。对一般区间 $[a,b]$，先使用线性映射

$$
t=\frac{2x-(a+b)}{b-a}
$$

把问题转到标准区间，再进行展开。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from numpy.polynomial import chebyshev as C
from numpy.polynomial import legendre as L

from py_sc import (
    chebyshev_fit_function,
    chebyshev_series_eval,
    legendre_fit_function,
    legendre_series_eval,
)


## Chebyshev 多项式

Chebyshev 多项式定义为

$$
T_n(x)=\cos(n\arccos x),\qquad x\in[-1,1].
$$

令 $x=\cos\theta$，则

$$
T_n(\cos\theta)=\cos(n\theta).
$$

由三角恒等式可以得到递推关系

$$
T_0(x)=1,\qquad T_1(x)=x,\qquad T_{n+1}(x)=2xT_n(x)-T_{n-1}(x).
$$

前几个多项式是

$$
T_0=1,\quad T_1=x,\quad T_2=2x^2-1,\quad T_3=4x^3-3x.
$$

Chebyshev 多项式在 $[-1,1]$ 上满足 $|T_n(x)|\le 1$，并且具有等幅振荡特征。这种特征和最佳一致逼近中的误差交错思想密切相关。

下面先在 notebook 中写一个教学版实现，展示递推逻辑。


In [ ]:
def teaching_chebyshev_values(x, degree):
    x = np.asarray(x, dtype=float)
    values = np.empty((x.size, degree + 1), dtype=float)
    values[:, 0] = 1.0
    if degree >= 1:
        values[:, 1] = x
    for n in range(1, degree):
        values[:, n + 1] = 2 * x * values[:, n] - values[:, n - 1]
    return values

x_plot = np.linspace(-1.0, 1.0, 500)
values = teaching_chebyshev_values(x_plot, 5)

plt.figure(figsize=(8, 4.5))
for n in range(6):
    plt.plot(x_plot, values[:, n], label=f"T_{n}")
plt.title("Chebyshev 多项式的递推生成")
plt.xlabel("x")
plt.ylabel("T_n(x)")
plt.legend(ncol=3)
plt.tight_layout()
plt.show()


## Chebyshev 级数逼近的教学式实现

我们希望把函数写成

$$
f(x)\approx \sum_{k=0}^n c_kT_k(x).
$$

理论上，Chebyshev 多项式关于权函数

$$
w(x)=\frac{1}{\sqrt{1-x^2}}
$$

正交；令 $x=\cos\theta$ 后，Chebyshev 展开也可以理解为余弦展开。

本节为了教学清晰，采用矩阵方式计算系数：先在 Chebyshev 零点上采样，再建立 Chebyshev Vandermonde 矩阵

$$
V_{ij}=T_j(x_i),
$$

然后求解或最小二乘求系数

$$
Vc\approx f(x_i).
$$

这不是最高效的 Chebyshev 变换，但每一步都能看见基函数矩阵、采样点和系数之间的关系。


In [ ]:
def teaching_chebyshev_fit(func, degree, sample_count=None):
    if sample_count is None:
        sample_count = degree + 1
    k = np.arange(sample_count)
    nodes = np.cos((2 * k + 1) * np.pi / (2 * sample_count))
    values = func(nodes)
    vandermonde = teaching_chebyshev_values(nodes, degree)
    coefficients, *_ = np.linalg.lstsq(vandermonde, values, rcond=None)
    return coefficients


def target(x):
    return np.exp(x) * np.cos(3 * x)

x_eval = np.linspace(-1.0, 1.0, 600)
y_true = target(x_eval)

teaching_coefficients = teaching_chebyshev_fit(target, degree=12, sample_count=40)
y_teaching = C.chebval(x_eval, teaching_coefficients)

# 对照 src/py_sc/approximation.py 中的可复用版本。
formal_coefficients = chebyshev_fit_function(target, degree=12, sample_count=40)
y_formal = chebyshev_series_eval(formal_coefficients, x_eval)

print("教学版和正式版系数最大差异:", np.max(np.abs(teaching_coefficients - formal_coefficients)))
print("Chebyshev 逼近最大误差:", np.max(np.abs(y_true - y_formal)))


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(x_eval, y_true, label="目标函数")
plt.plot(x_eval, y_formal, "--", label="Chebyshev 12 次逼近")
plt.semilogy([], [])
plt.title("Chebyshev 级数逼近光滑函数")
plt.xlabel("x")
plt.ylabel("函数值")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.5))
plt.semilogy(x_eval, np.abs(y_true - y_formal) + 1e-16)
plt.title("Chebyshev 逼近逐点误差")
plt.xlabel("x")
plt.ylabel("绝对误差")
plt.tight_layout()
plt.show()


## Legendre 级数逼近

Legendre 多项式关于普通内积正交：

$$
\int_{-1}^{1}P_i(x)P_j(x)\,dx=0,\qquad i\neq j.
$$

并且

$$
\int_{-1}^{1}P_n^2(x)\,dx=\frac{2}{2n+1}.
$$

若

$$
p_n(x)=\sum_{k=0}^n a_kP_k(x)
$$

是平方意义下的最佳逼近，则误差 $f-p_n$ 必须和每个 $P_j$ 正交：

$$
\langle f-p_n,P_j\rangle=0.
$$

利用正交性得到投影系数

$$
a_j=\frac{\langle f,P_j\rangle}{\langle P_j,P_j\rangle}
=\frac{2j+1}{2}\int_{-1}^{1}f(x)P_j(x)\,dx.
$$

下面用 Gauss-Legendre 求积做一个教学版投影实现。这个实现强调“积分内积 -> 投影系数”的过程，而不是直接调用黑箱拟合接口。


In [ ]:
def teaching_legendre_fit(func, degree, quadrature_order=80):
    nodes, weights = L.leggauss(quadrature_order)
    y = func(nodes)
    vandermonde = L.legvander(nodes, degree)
    coefficients = np.empty(degree + 1)
    for n in range(degree + 1):
        coefficients[n] = 0.5 * (2 * n + 1) * np.sum(weights * y * vandermonde[:, n])
    return coefficients

teaching_legendre = teaching_legendre_fit(target, degree=12, quadrature_order=80)
formal_legendre = legendre_fit_function(target, degree=12, quadrature_order=80)
y_legendre = legendre_series_eval(formal_legendre, x_eval)

print("教学版和正式版 Legendre 系数最大差异:", np.max(np.abs(teaching_legendre - formal_legendre)))
print("Legendre 逼近最大误差:", np.max(np.abs(y_true - y_legendre)))


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.semilogy(x_eval, np.abs(y_true - y_formal) + 1e-16, label="Chebyshev 误差")
plt.semilogy(x_eval, np.abs(y_true - y_legendre) + 1e-16, label="Legendre 误差")
plt.title("Chebyshev 与 Legendre 逼近误差对比")
plt.xlabel("x")
plt.ylabel("绝对误差")
plt.legend()
plt.tight_layout()
plt.show()


## 小结

* Chebyshev 多项式可由三角定义或递推关系得到。
* Chebyshev 级数可以看作余弦展开在多项式区间上的表现。
* Chebyshev 零点采样在本章服务于函数逼近，而不是重复第二章插值公式。
* Legendre 级数来自普通平方范数下的正交投影。
* Chebyshev 更强调端点控制和区间逼近稳定性；Legendre 更直接对应 $L^2$ 投影。
* Notebook 中先展示教学式实现，再调用 `py_sc` 中的正式函数做对照。
